In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, precision_recall_curve, f1_score
from joblib import Parallel, delayed
import matplotlib.pyplot as plt

In [3]:
# ─── 1. Utilities & Preprocessing ─────────────────────────────────────────────
lam, p, niter = 1e4, 0.01, 10

def baseline_als(y):
    L = len(y)
    D = np.diff(np.eye(L), 2)
    D = lam * D.dot(D.T)
    w = np.ones(L)
    for _ in range(niter):
        b = np.linalg.solve(np.diag(w) + D, w * y)
        w = p * (y > b) + (1 - p) * (y < b)
    return b

def preprocess_raman_single(spectrum):
    b = baseline_als(spectrum)
    c = spectrum - b
    norm = np.linalg.norm(c)
    out = c / norm if norm > 0 else c
    return np.abs(out)

def preprocess_fft_single(spectrum):
    b = baseline_als(spectrum)
    c = spectrum - b
    fft_vals = np.fft.rfft(c)
    mag = np.abs(fft_vals)
    mag = np.log1p(mag)
    norm = np.linalg.norm(mag)
    out = mag / norm if norm > 0 else mag
    return out

def floatify_cols(df):
    new = []
    for c in df.columns:
        if c in ('Label', 'Label 1', 'Label 2'):
            new.append(c)
        else:
            new.append(float(c))
    df.columns = new

In [4]:
# ─── 2. Model Definitions (Must match saved weights) ──────────────────────────
class SiameseNet(nn.Module):
    def __init__(self, input_len, embed_dim=64):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv1d(1,16,7,padding=3), nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(16,32,5,padding=2), nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Flatten(),
            nn.Linear((input_len//4)*32, embed_dim),
            nn.ReLU()
        )
    def forward(self,x):
        z = self.encoder(x)
        return F.normalize(z, dim=1)

class PresenceNetLogits(nn.Module):
    def __init__(self, D, C):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(D, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, C)
        )
    def forward(self, x):
        return self.net(x)

In [5]:
# ─── 3. Load Resources & Initialize ───────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Running on: {device}")

# A) Load Reference to get Classes
ref_df = pd.read_csv('reference_v2.csv')
floatify_cols(ref_df)
ref_labels = ref_df['Label'].values
classes = sorted(np.unique(ref_labels))
C = len(classes)
class_to_i = {c:i for i,c in enumerate(classes)}
print(f"Loaded {C} classes.")

# B) Initialize Models
# Note: Input lengths must match your data. 
# Assuming standard Raman length (e.g. 1000 or however many points your CSV has)
# We will check input length when we load the real data below.

# Placeholder initialization - we will resize layers if needed or rely on data shape
# Ideally, we read one row of data first to set input_len
mix_df = pd.read_csv('mixtures_dataset.csv') # Load now to get shapes
floatify_cols(mix_df)
wav_cols = [c for c in mix_df.columns if c not in ['Label 1', 'Label 2']]
input_len_raman = len(wav_cols)
# FFT length calculation (rfft outputs n/2 + 1)
input_len_fft = input_len_raman // 2 + 1 

siamese_raman = SiameseNet(input_len=input_len_raman, embed_dim=64).to(device)
siamese_fft = SiameseNet(input_len=input_len_fft, embed_dim=64).to(device)
model_boost = PresenceNetLogits(D=128, C=C).to(device)

# C) Load Weights
print("Loading weights...")
siamese_raman.load_state_dict(torch.load('siamese_mixture.pth', map_location=device))
siamese_fft.load_state_dict(torch.load('siamese_mixture_fft.pth', map_location=device))
checkpoint = torch.load('presence_net_logits.pth', map_location=device)
model_boost.load_state_dict(checkpoint['model_state_dict'])

siamese_raman.eval()
siamese_fft.eval()
model_boost.eval()
print("Models loaded successfully.")

Running on: cuda
Loaded 12 classes.
Loading weights...
Models loaded successfully.


In [6]:
# ─── 4. Process Real Data (Embeddings) ────────────────────────────────────────
print(f"Processing {len(mix_df)} real mixture samples...")

real_specs = mix_df[wav_cols].values

# Parallel Preprocessing
print("... running Raman preprocessing")
real_raman = np.vstack(
    Parallel(n_jobs=-1)(delayed(preprocess_raman_single)(s) for s in real_specs)
)
print("... running FFT preprocessing")
real_fft = np.vstack(
    Parallel(n_jobs=-1)(delayed(preprocess_fft_single)(s) for s in real_specs)
)

# Extract Embeddings
print("... extracting embeddings")
batch_size = 256
dataset_raw = TensorDataset(
    torch.tensor(real_raman, dtype=torch.float32), 
    torch.tensor(real_fft, dtype=torch.float32)
)
loader_raw = DataLoader(dataset_raw, batch_size=batch_size, shuffle=False)

embeds_list = []
with torch.no_grad():
    for xb_r, xb_f in loader_raw:
        xb_r, xb_f = xb_r.to(device).unsqueeze(1), xb_f.to(device).unsqueeze(1)
        emb_r = siamese_raman(xb_r)
        emb_f = siamese_fft(xb_f)
        concat = torch.cat([emb_r, emb_f], dim=1)
        embeds_list.append(concat.cpu().numpy())

X_real = np.vstack(embeds_list) # (N, 128)

# Build Labels
print("... building ground truth")
Y_real = np.zeros((len(mix_df), C), dtype=int)
for idx, row in mix_df.iterrows():
    l1, l2 = row['Label 1'], row['Label 2']
    if l1 in class_to_i: Y_real[idx, class_to_i[l1]] = 1
    if l2 in class_to_i: Y_real[idx, class_to_i[l2]] = 1

Processing 580 real mixture samples...
... running Raman preprocessing
... running FFT preprocessing
... extracting embeddings
... building ground truth


In [7]:
# ─── 5. Baseline: Full Data @ Default Threshold ───────────────────────────────
print("\n" + "="*50)
print("BASELINE: Real Data (100%) with Default Threshold 0.5")
print("="*50)

with torch.no_grad():
    logits_all = model_boost(torch.tensor(X_real, dtype=torch.float32, device=device))
    probs_all = torch.sigmoid(logits_all).cpu().numpy()

y_pred_default = (probs_all >= 0.5).astype(int)

# Filter out classes that don't exist in the real set to avoid cluttered report
supports = Y_real.sum(axis=0)
valid_idx = np.where(supports > 0)[0]
valid_names = [classes[i] for i in valid_idx]

print(classification_report(
    Y_real[:, valid_idx], 
    y_pred_default[:, valid_idx], 
    target_names=valid_names, 
    zero_division=0
))


BASELINE: Real Data (100%) with Default Threshold 0.5
                       precision    recall  f1-score   support

      1-dodecanethiol       0.87      0.99      0.92       243
 6-mercapto-1-hexanol       0.94      0.85      0.89       108
              benzene       1.00      1.00      1.00       193
         benzenethiol       0.67      1.00      0.80        72
                 etoh       1.00      1.00      1.00       121
                 meoh       1.00      0.99      0.99       243
n,n-dimethylformamide       0.99      1.00      0.99        72
             pyridine       1.00      1.00      1.00       108

            micro avg       0.94      0.98      0.96      1160
            macro avg       0.93      0.98      0.95      1160
         weighted avg       0.95      0.98      0.96      1160
          samples avg       0.95      0.98      0.96      1160



In [8]:
# ─── 6. Calibration Split (25% Calib / 75% Test) ──────────────────────────────
# We use a random split. 
# X_cal, Y_cal  -> used to find best thresholds
# X_test, Y_test -> used to report final performance

X_cal, X_test, Y_cal, Y_test, probs_cal, probs_test = train_test_split(
    X_real, Y_real, probs_all, test_size=0.75, random_state=42
)

print("\n" + "="*50)
print(f"CALIBRATION START: Using {len(X_cal)} samples to tune, {len(X_test)} to test.")
print("="*50)


CALIBRATION START: Using 145 samples to tune, 435 to test.


In [9]:
# ─── 7. Tune Thresholds on Calibration Set ────────────────────────────────────
best_thresholds = np.ones(C) * 0.5 # Default to 0.5
calibration_report = []

for k in range(C):
    # Only tune if this class exists in the calibration set
    if Y_cal[:, k].sum() < 1:
        continue
        
    y_true_k = Y_cal[:, k]
    y_probs_k = probs_cal[:, k]
    
    # Grid search for best F1 score
    precisions, recalls, thresholds = precision_recall_curve(y_true_k, y_probs_k)
    
    # Calculate F1 for each threshold
    f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-12)
    
    # Find max F1
    best_idx = np.argmax(f1_scores)
    
    # Handle edge case where thresholds array is shorter than p/r arrays
    if best_idx < len(thresholds):
        best_thr = thresholds[best_idx]
    else:
        best_thr = 0.5
        
    best_thresholds[k] = best_thr
    
    calibration_report.append({
        "Class": classes[k],
        "Best_Thr": best_thr,
        "Calib_F1": f1_scores[best_idx],
        "Support_Calib": int(y_true_k.sum())
    })

cal_df = pd.DataFrame(calibration_report)
print("\nTop 5 Classes by Calibration F1 (and their new thresholds):")
print(cal_df.sort_values("Calib_F1", ascending=False).head(5))

print("\nBottom 5 Classes by Calibration F1:")
print(cal_df.sort_values("Calib_F1", ascending=True).head(5))


Top 5 Classes by Calibration F1 (and their new thresholds):
                   Class  Best_Thr  Calib_F1  Support_Calib
3           benzenethiol  0.994808       1.0             19
2                benzene  1.000000       1.0             43
5                   meoh  0.621263       1.0             61
4                   etoh  0.863869       1.0             29
6  n,n-dimethylformamide  1.000000       1.0             19

Bottom 5 Classes by Calibration F1:
                  Class  Best_Thr  Calib_F1  Support_Calib
0       1-dodecanethiol  0.268966  0.924242             61
1  6-mercapto-1-hexanol  0.485002  0.981818             27
2               benzene  1.000000  1.000000             43
3          benzenethiol  0.994808  1.000000             19
4                  etoh  0.863869  1.000000             29


In [10]:
# ─── 8. Final Test on Remaining 75% ───────────────────────────────────────────
print("\n" + "="*50)
print("FINAL TEST: Evaluating on Remainder (75%) using Calibrated Thresholds")
print("="*50)

# Apply specific threshold per class
Y_pred_tuned = np.zeros_like(probs_test, dtype=int)
for k in range(C):
    Y_pred_tuned[:, k] = (probs_test[:, k] >= best_thresholds[k]).astype(int)

# Filter for valid classes in the test set
supports_test = Y_test.sum(axis=0)
valid_idx_test = np.where(supports_test > 0)[0]
valid_names_test = [classes[i] for i in valid_idx_test]

print(classification_report(
    Y_test[:, valid_idx_test], 
    Y_pred_tuned[:, valid_idx_test], 
    target_names=valid_names_test, 
    zero_division=0
))


FINAL TEST: Evaluating on Remainder (75%) using Calibrated Thresholds
                       precision    recall  f1-score   support

      1-dodecanethiol       0.87      0.99      0.93       182
 6-mercapto-1-hexanol       0.94      0.93      0.93        81
              benzene       1.00      1.00      1.00       150
         benzenethiol       1.00      0.98      0.99        53
                 etoh       1.00      0.98      0.99        92
                 meoh       1.00      0.98      0.99       182
n,n-dimethylformamide       1.00      1.00      1.00        53
             pyridine       1.00      1.00      1.00        77

            micro avg       0.97      0.98      0.97       870
            macro avg       0.98      0.98      0.98       870
         weighted avg       0.97      0.98      0.98       870
          samples avg       0.97      0.98      0.97       870



In [11]:
# ─── 9. Comparison: Baseline (0.5) vs Tuned on Test Set ───────────────────────
# To see the lift, let's run the default 0.5 on the Test set specifically
Y_pred_default_test = (probs_test >= 0.5).astype(int)

f1_def = f1_score(Y_test, Y_pred_default_test, average='weighted', zero_division=0)
f1_tun = f1_score(Y_test, Y_pred_tuned, average='weighted', zero_division=0)

print(f"\nSummary of Improvement on Test Set (Weighted F1):")
print(f"Default (0.5): {f1_def:.4f}")
print(f"Calibrated:    {f1_tun:.4f}")
print(f"Delta:         {f1_tun - f1_def:+.4f}")


Summary of Improvement on Test Set (Weighted F1):
Default (0.5): 0.9589
Calibrated:    0.9751
Delta:         +0.0162


In [12]:
import json

# 1. Create a dictionary mapping class names to their new thresholds
threshold_dict = {
    class_name: float(thr) 
    for class_name, thr in zip(classes, best_thresholds)
}

# 2. Save to JSON
with open('calibrated_thresholds.json', 'w') as f:
    json.dump(threshold_dict, f, indent=4)

print("Saved thresholds to calibrated_thresholds.json")

Saved thresholds to calibrated_thresholds.json
